Q1—Créer la SparkSession

In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("TradeCorp ETL") \
    .getOrCreate()


Q2—Lire les 8 CSV

In [2]:
chemin = "/home/jovyan/data/"

df_categories = spark.read.csv(chemin + "categories.csv",header=True, inferSchema=True)
df_customers = spark.read.csv(chemin + "customers.csv",header=True, inferSchema=True)
df_employees = spark.read.csv(chemin + "employees.csv",header=True, inferSchema=True)
df_order_details = spark.read.csv(chemin + "order_details.csv",header=True, inferSchema=True)
df_orders = spark.read.csv(chemin + "orders.csv",header=True, inferSchema=True)
df_products = spark.read.csv(chemin + "products.csv",header=True, inferSchema=True)
df_shippers = spark.read.csv(chemin + "shippers.csv",header=True, inferSchema=True)
df_suppliers = spark.read.csv(chemin + "suppliers.csv",header=True, inferSchema=True)



Q3—Explorer le schema

In [3]:
print("--- Schéma de df_categories ---")
df_categories.printSchema()

print("--- Schéma de df_customers ---")
df_customers.printSchema()

print("--- Schéma de df_employees ---")
df_employees.printSchema()

print("--- Schéma de df_order_details ---")
df_order_details.printSchema()

print("--- Schéma de df_orders ---")
df_orders.printSchema()

print("--- Schéma de df_products ---")
df_products.printSchema()

print("--- Schéma de df_shippers ---")
df_shippers.printSchema()

print("--- Schéma de df_suppliers ---")
df_suppliers.printSchema()

--- Schéma de df_categories ---
root
 |-- category_id: integer (nullable = true)
 |-- category_name: string (nullable = true)
 |-- description: string (nullable = true)
 |-- picture: string (nullable = true)

--- Schéma de df_customers ---
root
 |-- customer_id: string (nullable = true)
 |-- company_name: string (nullable = true)
 |-- contact_name: string (nullable = true)
 |-- contact_title: string (nullable = true)
 |-- address: string (nullable = true)
 |-- city: string (nullable = true)
 |-- region: string (nullable = true)
 |-- postal_code: string (nullable = true)
 |-- country: string (nullable = true)
 |-- phone: string (nullable = true)
 |-- fax: string (nullable = true)

--- Schéma de df_employees ---
root
 |-- employee_id: integer (nullable = true)
 |-- last_name: string (nullable = true)
 |-- first_name: string (nullable = true)
 |-- title: string (nullable = true)
 |-- title_of_courtesy: string (nullable = true)
 |-- birth_date: date (nullable = true)
 |-- hire_date: date (

Q4—Afficher les données

In [4]:
print("--- Les 5 premières lignes ---")
df_categories.show(5)

print("--- Les 5 premières lignes ---")
df_customers.show(5)

print("--- Les 5 premières lignes ---")
df_employees.show(5)

print("--- Les 5 premières lignes ---")
df_order_details.show(5)

print("--- Les 5 premières lignes ---")
df_orders.show(5)

print("--- Les 5 premières lignes ---")
df_products.show(5)

print("--- Les 5 premières lignes ---")
df_shippers.show(5)

print("--- Les 5 premières lignes ---")
df_suppliers.show(5)



--- Les 5 premières lignes ---
+-----------+--------------+--------------------+-------+
|category_id| category_name|         description|picture|
+-----------+--------------+--------------------+-------+
|          1|     Beverages|Soft drinks, coff...|   NULL|
|          2|    Condiments|Sweet and savory ...|   NULL|
|          3|   Confections|Desserts, candies...|   NULL|
|          4|Dairy Products|             Cheeses|   NULL|
|          5|Grains/Cereals|Breads, crackers,...|   NULL|
+-----------+--------------+--------------------+-------+
only showing top 5 rows
--- Les 5 premières lignes ---
+-----------+--------------------+------------------+--------------------+--------------------+-----------+------+-----------+-------+--------------+--------------+
|customer_id|        company_name|      contact_name|       contact_title|             address|       city|region|postal_code|country|         phone|           fax|
+-----------+--------------------+------------------+---------

Observations Q4: on peut voir que Spark range les données un peu n'importe comment , ce n'est pas bien structuré . 
On remarque aussi que les données sont sales avec des valeurs manquantes , il y a des NULL.

Q5—Compter les lignes

In [5]:
# création d'une liste 
comptage_lignes = [
    ("categories", df_categories.count()),
    ("customers", df_customers.count()),
    ("employees", df_employees.count()),
    ("order_details", df_order_details.count()),
    ("orders", df_orders.count()),
    ("products", df_products.count()),
    ("shippers", df_shippers.count()),
    ("suppliers", df_suppliers.count())
 ]
df_tableau = spark.createDataFrame(comptage_lignes, ["Tables", "Nb_lignes"])
df_tableau.show()


+-------------+---------+
|       Tables|Nb_lignes|
+-------------+---------+
|   categories|        8|
|    customers|       91|
|    employees|        9|
|order_details|     2155|
|       orders|      830|
|     products|       77|
|     shippers|        6|
|    suppliers|       29|
+-------------+---------+



Q6—Statistiquesdescriptives 
 Sur df_orders et df_products essayer d’obtenir les statistiques suivantes:min,max,mean,stddev.

In [6]:
# stats descriptives pour df_orders
print("--- Statistiques pour df_orders---")
df_orders.describe().show()

# stats descriptives pour df_products
print("--- Statistiques pour df_products---")
df_products.describe().show()


--- Statistiques pour df_orders---
+-------+-----------------+-----------+------------------+------------------+------------------+--------------------+--------------------+---------+-----------+------------------+------------+
|summary|         order_id|customer_id|       employee_id|          ship_via|           freight|           ship_name|        ship_address|ship_city|ship_region|  ship_postal_code|ship_country|
+-------+-----------------+-----------+------------------+------------------+------------------+--------------------+--------------------+---------+-----------+------------------+------------+
|  count|              830|        830|               830|               830|               830|                 830|                 830|      830|        323|               811|         830|
|   mean|          10662.5|       NULL| 4.403614457831325|2.0072289156626506| 78.24420481927719|                NULL|                NULL|     NULL|       NULL|39975.067357512955|        NULL|


Q7—Lazy evaluation

L'évaluation paresseuse signifie que Spark n'exécute aucun code tant qu'on ne l'oblige pas explicitement (en appelant une action ).
Au lieu de cela, il enregistre chaque étape que l'on écrit dans un « plan logique » pour une optimisation ultérieure.

Dans Spark, les transformations sont des opérations sur un RDD (Resilient Distributed Dataset) ou un DataFrame qui définissent 
un nouveau jeu de données à partir du jeu existant. Ces transformations sont paresseuses , c'est-à-dire qu'elles ne s'exécutent pas immédiatement.
En exemple, nous pouvons citer les commandes filter, select et map.

Les actions,elles,déclenchent l'exécution effective des transformations qui ont été définies. 
Elles renvoient des valeurs ou écrivent des données quelque part.
En exemple, nous pouvons citer count, collect, show.

In [ ]:
Q8—SparkUI

Un stage représente une séquence de transformations exécutables en une seule passe, c'est-à-dire sans réorganisation des données.
Une tâche est la plus petite unité de travail pouvant être planifiée. Chaque étape est divisée en tâches. 
Une tâche est une unité d'exécution qui s'exécute sur une seule machine. 

In [ ]:
Q9—Spark vs Pandas

In [ ]:
import pandas as pd
import time 
print("--- Comparaison de vitesse de lecture entre Pandas et Spark ---")
depart_pandas = time.time()
df_pandas = pd.read_csv("/home/jovyan/data/orders.csv")
fin_pandas = time.time()

print(f"Durée Pandas : {fin_pandas - depart_pandas:.3f}secondes")

depart_spark = time.time()
df_spark = spark.read.csv("/home/jovyan/data/orders.csv", header=True, inferSchema=True)
fin_spark = time.time()

print(f"Durée Spark : {fin_spark - depart_spark:.3f}secondes")




On remarque que Pandas a été 3 fois plus rapide que Spark , il gère mieux les petits volumes. A contrario, Spark lui ne sait pas gérer
les petits volumes sachant qu'il perd beaucoup de temps à structurer son environnement , il est parfait pour le Big Data.

In [ ]:
Q10—Colonnes et types

In [9]:
print("---Liste de colonnes de df_orders---")
print(df_orders.columns)

print("---Affichage des types---")
print(df_orders.dtypes)

---Liste de colonnes de df_orders---
['order_id', 'customer_id', 'employee_id', 'order_date', 'required_date', 'shipped_date', 'ship_via', 'freight', 'ship_name', 'ship_address', 'ship_city', 'ship_region', 'ship_postal_code', 'ship_country']
---Affichage des types---
[('order_id', 'int'), ('customer_id', 'string'), ('employee_id', 'int'), ('order_date', 'date'), ('required_date', 'date'), ('shipped_date', 'date'), ('ship_via', 'int'), ('freight', 'double'), ('ship_name', 'string'), ('ship_address', 'string'), ('ship_city', 'string'), ('ship_region', 'string'), ('ship_postal_code', 'string'), ('ship_country', 'string')]


On peut observer à travers cet affichage que tout est très bien typé , les données initiales sont plutôt propre.